In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR.parent / "data" / "processed"
REPORTS_DIR = PROJECT_DIR / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", PROCESSED_DIR)
print("Reports folder:", REPORTS_DIR)

In [ ]:
customers = pd.read_parquet(PROCESSED_DIR / "customers.parquet")
orders = pd.read_parquet(PROCESSED_DIR / "orders.parquet")
order_items = pd.read_parquet(PROCESSED_DIR / "order_items.parquet")
products = pd.read_parquet(PROCESSED_DIR / "products.parquet")
transactions = pd.read_parquet(PROCESSED_DIR / "transactions.parquet")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Transactions:", transactions.shape)

In [ ]:
print("Files inside processed folder:")

for file in sorted(PROCESSED_DIR.iterdir()):
    print(file.name)

In [ ]:
customers = pd.read_parquet(PROCESSED_DIR / "customers_clean.parquet")
orders = pd.read_parquet(PROCESSED_DIR / "orders_clean.parquet")
order_items = pd.read_parquet(PROCESSED_DIR / "order_items_clean.parquet")
products = pd.read_parquet(PROCESSED_DIR / "products_clean.parquet")
transactions = pd.read_parquet(PROCESSED_DIR / "transactions_clean.parquet")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Transactions:", transactions.shape)

In [ ]:
print("Customers columns:")
print(customers.columns.tolist())

print("\nOrders columns:")
print(orders.columns.tolist())

print("\nOrder items columns:")
print(order_items.columns.tolist())

print("\nProducts columns:")
print(products.columns.tolist())

print("\nTransactions columns:")
print(transactions.columns.tolist())

In [ ]:
print("Order statuses:")
print(orders["status"].value_counts(dropna=False))

print("\nPayment statuses:")
print(transactions["payment_status"].value_counts(dropna=False))

In [ ]:
paid_transactions = transactions[
    transactions["payment_status"].str.lower() == "paid"
].copy()

completed_orders = orders[
    orders["status"].str.lower() == "completed"
].copy()

total_revenue = paid_transactions["amount"].sum()
successful_orders = paid_transactions["order_id"].nunique()

average_order_value = (
    total_revenue / successful_orders
    if successful_orders > 0
    else 0
)

total_customers = customers["customer_id"].nunique()

print(f"Total revenue: {total_revenue:,.2f}")
print(f"Successful paid orders: {successful_orders}")
print(f"Average order value: {average_order_value:,.2f}")
print(f"Total customers: {total_customers}")

In [ ]:
orders_per_customer = (
    orders.groupby("customer_id")["order_id"]
    .nunique()
)

returning_customers = (orders_per_customer > 1).sum()
customers_with_orders = orders_per_customer.shape[0]

returning_customer_rate = (
    returning_customers / customers_with_orders * 100
    if customers_with_orders > 0
    else 0
)

print(f"Customers with orders: {customers_with_orders}")
print(f"Returning customers: {returning_customers}")
print(f"Returning customer rate: {returning_customer_rate:.2f}%")

In [ ]:
print("Files inside the processed folder:")

for file in sorted(PROCESSED_DIR.iterdir()):
    print(file.name)

In [ ]:
import psycopg2

DB_NAME = "intern_db"
DB_USER = "postgres"
DB_PASSWORD = "Abdulla11-11"
DB_HOST = "localhost"
DB_PORT = "5432"

connection = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)

prediction_query = """
    SELECT
        customer_id,
        prediction,
        probability,
        prediction_timestamp
    FROM customer_predictions
    ORDER BY customer_id;
"""

predictions = pd.read_sql(prediction_query, connection)

connection.close()

print("Predictions shape:", predictions.shape)
predictions.head()

In [ ]:
predicted_reorders = (predictions["prediction"] == 1).sum()
predicted_non_reorders = (predictions["prediction"] == 0).sum()

predicted_reorder_rate = (
    predicted_reorders / len(predictions) * 100
    if len(predictions) > 0
    else 0
)

average_reorder_probability = predictions["probability"].mean() * 100

print(f"Customers predicted to reorder: {predicted_reorders}")
print(f"Customers predicted not to reorder: {predicted_non_reorders}")
print(f"Predicted reorder rate: {predicted_reorder_rate:.2f}%")
print(
    f"Average reorder probability: "
    f"{average_reorder_probability:.2f}%"
)

In [ ]:
print("Prediction counts:")
print(predictions["prediction"].value_counts(dropna=False))

print("\nProbability statistics:")
print(predictions["probability"].describe())

print("\nUnique probabilities:")
print(predictions["probability"].unique()[:20])

In [ ]:
kpi_summary = pd.DataFrame(
    {
        "KPI": [
            "Total revenue",
            "Successful paid orders",
            "Average order value",
            "Total customers",
            "Customers with orders",
            "Returning customers",
            "Returning customer rate",
            "Predicted reorder rate",
        ],
        "Value": [
            f"{total_revenue:,.2f}",
            successful_orders,
            f"{average_order_value:,.2f}",
            total_customers,
            customers_with_orders,
            returning_customers,
            f"{returning_customer_rate:.2f}%",
            f"{predicted_reorder_rate:.2f}%",
        ],
    }
)

kpi_summary

In [ ]:
sales_by_category = (
    order_items
    .merge(
        products[["product_id", "category"]],
        on="product_id",
        how="left"
    )
    .groupby("category", as_index=False)["total_price"]
    .sum()
    .sort_values("total_price", ascending=False)
)

sales_by_category

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    sales_by_category["category"],
    sales_by_category["total_price"]
)

plt.title("Sales Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=20)
plt.tight_layout()

category_chart_path = REPORTS_DIR / "sales_by_category.png"

plt.savefig(
    category_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Chart saved to:", category_chart_path)

In [ ]:
paid_transactions["transaction_date"] = pd.to_datetime(
    paid_transactions["transaction_date"]
)

daily_revenue = (
    paid_transactions
    .groupby(
        paid_transactions["transaction_date"].dt.date,
        as_index=False
    )["amount"]
    .sum()
    .rename(
        columns={
            "transaction_date": "date",
            "amount": "revenue"
        }
    )
)

daily_revenue.head()

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    daily_revenue["date"],
    daily_revenue["revenue"],
    marker="o"
)

plt.title("Daily Paid Revenue")
plt.xlabel("Transaction Date")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()

daily_revenue_chart_path = REPORTS_DIR / "daily_revenue.png"

plt.savefig(
    daily_revenue_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Chart saved to:", daily_revenue_chart_path)

In [ ]:
top_products = (
    order_items
    .merge(
        products[["product_id", "product_name"]],
        on="product_id",
        how="left"
    )
    .groupby("product_name", as_index=False)["total_price"]
    .sum()
    .sort_values("total_price", ascending=False)
    .head(10)
)

top_products

In [ ]:
top_products_chart = top_products.sort_values(
    "total_price",
    ascending=True
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_products_chart["product_name"],
    top_products_chart["total_price"]
)

plt.title("Top 10 Products by Sales Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()

top_products_chart_path = REPORTS_DIR / "top_products.png"

plt.savefig(
    top_products_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Chart saved to:", top_products_chart_path)

In [ ]:
prediction_counts = predictions["prediction"].value_counts().sort_index()

prediction_labels = {
    0: "Not predicted to reorder",
    1: "Predicted to reorder"
}

prediction_chart_data = pd.DataFrame({
    "prediction": prediction_counts.index,
    "count": prediction_counts.values
})

prediction_chart_data["label"] = prediction_chart_data[
    "prediction"
].map(prediction_labels)

prediction_chart_data

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    prediction_chart_data["label"],
    prediction_chart_data["count"]
)

plt.title("Customer Reorder Predictions")
plt.xlabel("Prediction")
plt.ylabel("Number of Customers")
plt.tight_layout()

prediction_chart_path = REPORTS_DIR / "reorder_predictions.png"

plt.savefig(
    prediction_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Chart saved to:", prediction_chart_path)

In [ ]:
top_category = sales_by_category.iloc[0]["category"]
top_category_revenue = sales_by_category.iloc[0]["total_price"]

top_product = top_products.iloc[0]["product_name"]
top_product_revenue = top_products.iloc[0]["total_price"]

analytics_prompt = f"""
You are a business analytics assistant.

Write a short weekly sales summary using the following results:

- Total revenue: {total_revenue:,.2f}
- Successful paid orders: {successful_orders}
- Average order value: {average_order_value:,.2f}
- Total customers: {total_customers}
- Returning customer rate: {returning_customer_rate:.2f}%
- Top product category: {top_category}
- Top category revenue: {top_category_revenue:,.2f}
- Top product: {top_product}
- Top product revenue: {top_product_revenue:,.2f}
- Predicted reorder rate: {predicted_reorder_rate:.2f}%

Mention:
1. The strongest sales result.
2. One customer insight.
3. One recommendation.
4. A warning that the reorder prediction is experimental because it was produced from a small synthetic dataset.

Keep the summary clear and concise.
"""

print(analytics_prompt)

In [ ]:
from google import genai

print("Google GenAI imported successfully.")

In [ ]:
import sys
import subprocess

print("Notebook Python:")
print(sys.executable)

print("\nGoogle GenAI package:")
subprocess.run(
    [sys.executable, "-m", "pip", "show", "google-genai"]
)

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "google-genai"
])

print("Installation completed.")

In [ ]:
from google import genai

print("Google GenAI imported successfully.")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

env_path = Path.cwd().parent.parent / '.env'
load_dotenv(env_path)

gemini_api_key = os.getenv('GEMINI_API_KEY')

if not gemini_api_key:
    raise ValueError('GEMINI_API_KEY was not found.')

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=analytics_prompt
)

weekly_summary = response.text
print(weekly_summary)
